# CPW 介质示例共面波导（CPW）是一种平面传输线，其回波路径位于同一层上，位于带状导体两侧的平面上。其特性阻抗取决于带状导体的宽度以及与回波平面的间隙，从而为设计者提供了两个自由度。CPW 的一大优势在于，由于信号和回波路径都位于同一侧，因此可以方便地与元件进行并联或串联连接，从而无需使用过孔。另一方面，大型共面平面需要占用大量的面积。共面波导具有准 TEM 模式，并且比微带线具有更低的色散。为了避免寄生槽线模式的传播，必须将带状导体两侧的平面连接在一起。这可以通过在 MMIC 上使用空气桥或在衬底背面添加一个导体平面，并使用过孔栅栏将所有平面连接在一起来实现。当使用导体背面时，如果衬底的高度较小且与共面平面的间隙较大，则该线路的行为可能会类似于微带线。![不带导体背面的共面波导结构的等距视图](coplanar_waveguide_structure.svg "不带导体背面的共面波导结构")

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru
from skrf.media import CPW

rf.stylely()

## 测量两条不同长度的 CPWG 传输线测量于 2017 年 3 月 21 日在安立士 MS46524B 20GHz VNA 上进行。设置是对频率进行线性扫描，范围从 1 MHz 到 10 GHz，共 10000 个点。输出功率为 0 dBm，中频带宽为 1kHz，未使用平均或平滑处理。CPWGxxx 是一种长为 L、宽为 W、顶部与接地层之间的间隙宽为 G 的厚铜共面波导，导体位于高为 H 的基板上，顶部和底部均有接地层。在传输线的两侧设置了紧密排列的过孔壁，顶部和底部的接地层通过多个过孔连接。| 名称 | L (mm) | W (mm) | G (mm) | H (mm) | T (um) | 基板 || :--- | ---: | ---: | ---: | ---: | ---: | :--- || CPW100 | 100 | 1.70 | 0.50 | 1.55 | 50 | FR-4 || CPW200 | 200 | 1.70 | 0.50 | 1.55 | 50 | FR-4 |电路板的铣削是通过机械方式进行的，侧壁角度为 45°。为了设计目的，假设介电常数的相对值约为 4.5。![MSL100 和 MSL200 插图，两者均为微带线，MSL200 的长度是 MSL100 的两倍](MSL_CPWG_100_200.jpg "MSL100 和 MSL200")

## 查看测量结果我们查看插入损耗的相位和幅度，以及回波损耗的相位和幅度。CPWG200的插入损耗是CPWG100的两倍，并且在高达6 GHz的频率范围内基本呈线性关系。相位也显示出两倍的差异。这两个结论都与CPWG200的长度是CPWG100的两倍这一事实相符。对于这两个网络，回波损耗的插入损耗和相位具有相似的数量级。在低频时，插入损耗低于-20 dB，这表明阻抗匹配良好。与CPWG100相比，CPWG200的幅度最小值之间的距离缩短了两倍，这也与物理长度差异相符。

In [ ]:
# Load raw measurements
TL100 = rf.Network('CPWG100.s2p')
TL200 = rf.Network('CPWG200.s2p')

# plot them all
plt.figure(figsize=(10, 10))
plt.suptitle('Raw measurements')
plt.subplot(2, 2, 1)
TL100.plot_s_db(0, 0)
TL200.plot_s_db(0, 0)
TL100.plot_s_db(1, 1)
TL200.plot_s_db(1, 1)
plt.subplot(2, 2, 2)
TL100.plot_s_deg(0, 0)
TL200.plot_s_deg(0, 0)
TL100.plot_s_deg(1, 1)
TL200.plot_s_deg(1, 1)
plt.subplot(2, 2, 3)
TL100.plot_s_db(1, 0)
TL200.plot_s_db(1, 0)
TL100.plot_s_db(0, 1)
TL200.plot_s_db(0, 1)
plt.subplot(2, 2, 4)
TL100.plot_s_deg(1, 0)
TL200.plot_s_deg(1, 0)
TL100.plot_s_deg(0, 1)
TL200.plot_s_deg(0, 1)

## 消除连接器的影响SMA 连接器会在信号路径中产生不连续性。波在发射区域需要通过一个复杂的、具有三维几何形状的同轴到平面过渡结构。通过将 100 毫米的线路分成两半，并将其从 200 毫米的线路中移除，连接器和 50 毫米的线路将在两侧被移除。这将消除连接器产生的影响。

In [ ]:
# deembedding using IEEEP370 impedance corrected 2xthru method
dm = IEEEP370_SE_NZC_2xThru(dummy_2xthru = TL100, name = '2xthru')
fix1 = dm.s_side1
fix1.name = 'FIX-1'
fix2 = dm.s_side2
fix2.name = 'FIX-2'
d_dut = dm.deembed(TL200)
d_dut.name = 'd_DUT'

# plot them all
plt.figure(figsize=(10, 10))
plt.suptitle('Connectors models')
plt.subplot(2, 2, 1)
fix1.plot_s_db(0, 0)
fix2.plot_s_db(0, 0)
plt.subplot(2, 2, 2)
fix1.plot_s_deg(0, 0)
fix2.plot_s_deg(0, 0)
plt.subplot(2, 2, 3)
fix1.plot_s_db(1, 0)
fix2.plot_s_db(1, 0)
plt.subplot(2, 2, 4)
fix1.plot_s_deg(1, 0)
fix2.plot_s_deg(1, 0)

## 比较去嵌入后的DUT结果与CPW介质模拟结果现在我们将去嵌入后的测量结果与CPW介质的模拟结果进行比较。εr和tanD值取自同一目录中名为“将微带线模型与测量结果关联”的notebook。相位和幅度的插入损耗一致性非常好。整体而言，回波损耗的幅度似乎是正确的，表明匹配良好。由于信号会变得很小，因此低于-20 dB的回波损耗难以测量。因此，我们不应期望在此处获得良好的形状一致性。

In [ ]:
ep_r = 4.421
tanD = 0.0167

cpw = CPW(frequency=d_dut.frequency, w = 1.7e-3, s = 0.5e-3, t = 50e-6, h = 1.55e-3,
        ep_r = ep_r, tand = tanD, rho = 1.7e-8, z0_port = 50., has_metal_backside = True)
l = cpw.line(d = 100.0e-3, unit = 'm')
l.name = 'model'

# plot them all
plt.figure(figsize=(10, 10))
plt.suptitle('Comparison deembedded measurement and simulation')
plt.subplot(2,2,1)
d_dut.plot_s_db(0,0)
l.plot_s_db(0,0,)
plt.subplot(2,2,2)
d_dut.plot_s_deg(0,0)
l.plot_s_deg(0,0)
plt.subplot(2,2,3)
d_dut.plot_s_db(1,0)
l.plot_s_db(1,0)
plt.subplot(2,2,4)
d_dut.plot_s_deg(1,0)
l.plot_s_deg(1,0)


## 检查去嵌入过程可以进行两个基本检查，以确保由 IEEEP370 2xthru 算法生成的侧面模型的的一致性。1. 绘制 2xthru、side1 和 side2 的时域响应。侧面应该看起来像是 2xthru 的两半。这里就是这种情况。绘制 fixture-dut-fixture 的时域响应，它应该具有与 2xthru 相同的左右形状。在这里，我们可以看到一个小的阻抗失配，这将导致 dut 形状产生跳变。绘制 dut，看看它是否看起来像 fixture-dut-fixture 的中间部分。在这里，除了 dut 左右两侧的小跳变之外，所有内容似乎都正常。2. 通过从 2xthru 中去除两个侧面模型，检查残差。残差插入幅度的范围应为 ± 0.1 dB，残差插入损耗相位应在 ± 1° 以内。在这里，在整个带宽范围内，这两个侧面都符合这个要求。

In [ ]:
# compute residuals
res = dm.deembed(TL100)
res.name = 'residuals'
res.s += 1e-15 # avoid numeric singularities

# extrapolate to dc for time step
TL100_dc = TL100.extrapolate_to_dc(kind='linear')
TL200_dc = TL200.extrapolate_to_dc(kind='linear')
fix1_dc = fix1.extrapolate_to_dc(kind='cubic')
fix2_dc = fix2.extrapolate_to_dc(kind='cubic')
d_dut_dc = d_dut.extrapolate_to_dc(kind='cubic')

# plot them all
# time domain
plt.figure(figsize=(8, 4))
plt.suptitle('Time domain reflexion step response (DC extrapolation)')
TL100_dc.plot_z_time_step(0, 0)
TL200_dc.plot_z_time_step(0, 0)
fix1_dc.plot_z_time_step(0, 0)
fix2_dc.plot_z_time_step(0, 0)
d_dut_dc.plot_z_time_step(0, 0)
plt.xlim(-2, 4)

# residuals frequency domain
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
res.plot_s_db(1,0)
plt.subplot(1, 2, 2)
res.plot_s_deg(1,0)